# Big Data Lab 1 - Exercise 2
Clustering: Group movies by genres


In [ ]:
# Exercise 0: Prepare Movie Data
import findspark
import os
import sys
os.environ["JAVA_HOME"] = "C:\\Program Files\\Microsoft\\jdk-11.0.16.101-hotspot"
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
findspark.init()
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, explode
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

# Initialize Spark session
spark = SparkSession.builder \
    .appName("BigData_Lab1") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1") \
    .getOrCreate()

kafka_bootstrap_servers = "localhost:30090,localhost:30091,localhost:30092"

# Define schemas
movies_schema = StructType([
    StructField("movieId", IntegerType(), True),
    StructField("title", StringType(), True),
    StructField("genres", StringType(), True)
])

ratings_schema = StructType([
    StructField("userId", IntegerType(), True),
    StructField("movieId", IntegerType(), True),
    StructField("rating", DoubleType(), True),
    StructField("timestamp", IntegerType(), True)
])

tags_schema = StructType([
    StructField("userId", IntegerType(), True),
    StructField("movieId", IntegerType(), True),
    StructField("tag", StringType(), True),
    StructField("timestamp", IntegerType(), True)
])

# Read movies from Kafka
df_movies_kafka = spark.read \
    .format("kafka") \
    .option("kafka.bootstrap.servers", kafka_bootstrap_servers) \
    .option("subscribe", "Lab1_movies") \
    .option("startingOffsets", "earliest") \
    .load()
df_movies = df_movies_kafka.selectExpr("CAST(value AS STRING)") \
    .select(from_json(col("value"), movies_schema).alias("data")) \
    .select("data.*")

# Read ratings from Kafka
df_ratings_kafka = spark.read \
    .format("kafka") \
    .option("kafka.bootstrap.servers", kafka_bootstrap_servers) \
    .option("subscribe", "Lab1_ratings") \
    .option("startingOffsets", "earliest") \
    .load()
df_ratings = df_ratings_kafka.selectExpr("CAST(value AS STRING)") \
    .select(from_json(col("value"), ratings_schema).alias("data")) \
    .select("data.*")

# Read tags from Kafka
df_tags_kafka = spark.read \
    .format("kafka") \
    .option("kafka.bootstrap.servers", kafka_bootstrap_servers) \
    .option("subscribe", "Lab1_tags") \
    .option("startingOffsets", "earliest") \
    .load()
df_tags = df_tags_kafka.selectExpr("CAST(value AS STRING)") \
    .select(from_json(col("value"), tags_schema).alias("data")) \
    .select("data.*")

# Show schemas and sample data
df_movies.printSchema()
df_ratings.printSchema()
df_tags.printSchema()



## Data Preparation


In [ ]:
from pyspark.sql.functions import collect_list, concat_ws, regexp_replace

# Group tags by movie
movie_tags = df_tags.groupBy("movieId").agg(concat_ws(" ", collect_list("tag")).alias("tags_text"))

# Join with movies
df_movie_features = df_movies.join(movie_tags, on="movieId", how="left").fillna({"tags_text": "", "genres": ""})
df_movie_features = df_movie_features.withColumn("genres_space", regexp_replace("genres", "\\|", " "))



## Pipeline and Training


In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import Tokenizer, CountVectorizer, IDF, StopWordsRemover, VectorAssembler
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

# Text processing
tokenizer_tags = Tokenizer(inputCol="tags_text", outputCol="tags_tokens")
remover_tags = StopWordsRemover(inputCol="tags_tokens", outputCol="tags_clean")
cv_tags = CountVectorizer(inputCol="tags_clean", outputCol="tags_tf", vocabSize=500)
idf_tags = IDF(inputCol="tags_tf", outputCol="tags_tfidf")

# Genre processing
tokenizer_genres = Tokenizer(inputCol="genres_space", outputCol="genres_tokens")
cv_genres = CountVectorizer(inputCol="genres_tokens", outputCol="genres_vec")

assembler = VectorAssembler(inputCols=["tags_tfidf", "genres_vec"], outputCol="features")

# Prepare features
pipeline_prep = Pipeline(stages=[tokenizer_tags, remover_tags, cv_tags, idf_tags, tokenizer_genres, cv_genres, assembler])
prep_model = pipeline_prep.fit(df_movie_features)
df_prepared = prep_model.transform(df_movie_features)

k_values = [6, 8, 10, 12]
evaluator = ClusteringEvaluator()

for k in k_values:
    print(f"\n{'='*40}\nEvaluating K = {k}\n{'='*40}")
    kmeans = KMeans(featuresCol="features", k=k, seed=42)
    model_k = kmeans.fit(df_prepared)
    predictions = model_k.transform(df_prepared)
    
    # Silhouette Score
    silhouette = evaluator.evaluate(predictions)
    print(f"Silhouette Score for K={k}: {silhouette}\n")
    
    # Extract terms and samples
    centers = model_k.clusterCenters()
    vocab = prep_model.stages[2].vocabulary + prep_model.stages[5].vocabulary
    
    for cluster_id, center in enumerate(centers):
        # Top 10 terms
        top_indices = center.argsort()[-10:][::-1]
        top_terms = [vocab[i] for i in top_indices]
        print(f"Cluster {cluster_id} top terms: {', '.join(top_terms)}")
        
        # Sample 10 movies
        print(f"Cluster {cluster_id} sample movies:")
        predictions.filter(col("prediction") == cluster_id).select("title", "genres").show(10, truncate=False)

